# VWAP Stdev Bands

__Algorithm:__ session-anchored VWAP with volume-weighted stdev bands → mean-reversion entry on a furthest-band breach → exit on return to the VWAP middle.

__Pattern:__
1. Compute a daily-anchored VWAP and five upper/lower bands at \
   `(1.28, 2.01, 2.51, 3.09, 4.01)` standard deviations (same defaults as the \
   TradingView "VWAP Stdev Bands v2 Mod" preset).
2. When close crosses **through the furthest band** (from inside → outside), enter \
   a counter-trend trade: LONG below the lower band, SHORT above the upper band.
3. Exit when the close returns to the VWAP middle.

__Features:__
- VWAP + stdev recomputed cumulatively *within* each session — bands reset every UTC day.
- Causal by construction: bar i only depends on bars `0..i` of the active session, \
  so look-ahead is structurally impossible (same contract enforced by the backtester).
- All five stdev multipliers, the session anchor, and the entry-band index are \
  exposed on StrategyConfig — no magic numbers in the strategy file.

__How the VWAP Bands Algorithm Determines Entry/Exit:__
- VWAP = Σ(hl2 · volume) / Σ(volume), reset at each session boundary (vwap_session).
- Stdev = √(max(Σ(hl2² · volume)/Σ(volume) − VWAP², 0)).
- Entry band = `vwap_band_devs[vwap_entry_band]` × stdev around VWAP (default = 4.01σ).
- **Long entry:**  `close_prev ≥ lower` AND `close_now < lower`.
- **Short entry:** `close_prev ≤ upper` AND `close_now > upper`.
- **Exit:** `close ≥ VWAP` (long) or `close ≤ VWAP` (short) — first touch wins.

## Configuration: automatic

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import StrategyConfig, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = StrategyConfig()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic StrategyConfig().
STRATEGY_OVERRIDES = {}      # e.g. {"vwap_entry_band": 2, "vwap_session": "D"}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

num_candles is ignored when start_time is set.

Two distinct modes:
1. Range mode — start_time is set. \
   The fetcher paginates the full [start_time, end_time] window (end defaults to now). \
   num_candles is unused.
2. Count mode — start_time is None. \
   The fetcher returns the most recent num_candles ending at end_time (or now).

If you want exactly N bars (candles), drop start_time/end_time. \
If you want the full month/year/..., leave start_time/end_time and delete the CANDLES argument (or leave it since it's ignored in this case).

## VWAP Stdev Bands

In [5]:
# Import VWAP Bands strategy
from engine.strategies import VWAPBandsStrategy

In [ ]:
# Backtest VWAP Bands strategy
strategy = VWAPBandsStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [7]:
# Per-trade dollar P&L — the engine's real figures (TradingConfig sizing +
# initial_equity), already on each Trade and saved to <strategy>_trades.csv.
import pandas as pd

trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])

net = result.final_equity - result.initial_equity
print(f"Final balance : ${result.final_equity:,.2f}")
print(f"Net profit    : ${net:+,.2f}")
print(f"Net return    : {result.total_return_pct:+.2f}%")
print(f"Max drawdown  : {result.max_drawdown_pct:.2f}%")
trades_pnl


─────────────────────────────────────────────
  Dollar P&L (starting $100.00)
─────────────────────────────────────────────
  #  1  short     -0.13  →  $99.87
  #  2  short     -0.51  →  $99.36
  #  3  short     +0.28  →  $99.64
  #  4  long      -0.04  →  $99.60
  #  5  short     -0.15  →  $99.45
  #  6  long      -0.05  →  $99.40
  #  7  short     +0.19  →  $99.59
  #  8  long      -0.01  →  $99.58
  #  9  long      -0.02  →  $99.56
  # 10  long      +0.97  →  $100.53
  # 11  long      +0.12  →  $100.65
  # 12  long      -0.09  →  $100.56
  # 13  short     -0.01  →  $100.55
  # 14  long      -0.13  →  $100.42
  # 15  long      -0.07  →  $100.35
  # 16  long      -0.31  →  $100.04
  # 17  long      -0.00  →  $100.03
  # 18  long      +0.10  →  $100.13
  # 19  long      -0.36  →  $99.77
  # 20  long      -0.51  →  $99.26
  # 21  long      -0.22  →  $99.04
  # 22  short     -0.31  →  $98.72
  # 23  long      -0.06  →  $98.66
  # 24  short     +1.62  →  $100.28
  # 25  short     -0.25  

In [ ]:
# VWAP Bands strategy chart (VWAP + all 5 band pairs overlay automatically)
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Band hit frequency

In [12]:
# How often each band is reached (intraday touch via high ≥ upper or low ≤ lower),
# plus how many long / short entries the strategy fires when that band is the trigger.
import pandas as pd
from dataclasses import replace

from engine.core import Direction, SignalAction

high = prepared["high"]
low = prepared["low"]
total = len(prepared)

# Per-band entry counts: re-run the strategy once with each band index as the trigger.
# Shorts come from the upper side, longs from the lower side.
entries_per_band: dict[int, tuple[int, int]] = {}
for k, _ in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    cfg_k = replace(STRATEGY_CONFIG, vwap_entry_band=k)
    strat_k = VWAPBandsStrategy(cfg_k)
    result_k = Backtester(strat_k, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)
    longs = sum(1 for t in result_k.trades if t.direction == Direction.LONG)
    shorts = sum(1 for t in result_k.trades if t.direction == Direction.SHORT)
    entries_per_band[k] = (longs, shorts)

rows = []
for k, mult in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    longs, shorts = entries_per_band[k]
    upper_hits = int((high >= prepared[f"vwap_upper_{k}"]).sum())
    lower_hits = int((low  <= prepared[f"vwap_lower_{k}"]).sum())
    rows.append({"side": "upper", "band": k, "stdev": mult,
                 "hits": upper_hits, "pct_of_bars": upper_hits / total,
                 "long_entries": 0, "short_entries": shorts})
    rows.append({"side": "lower", "band": k, "stdev": mult,
                 "hits": lower_hits, "pct_of_bars": lower_hits / total,
                 "long_entries": longs, "short_entries": 0})

band_hits = (
    pd.DataFrame(rows)
      .sort_values("hits", ascending=False)
      .reset_index(drop=True)
)
band_hits["pct_of_bars"] = (band_hits["pct_of_bars"] * 100).round(2)
band_hits

,side,band,stdev,hits,pct_of_bars,long_entries,short_entries
0,upper,0,1.28,834,28.95,0,67
1,lower,0,1.28,813,28.22,58,0
2,upper,1,2.01,354,12.29,0,52
3,lower,1,2.01,310,10.76,43,0
4,upper,2,2.51,181,6.28,0,30
5,lower,2,2.51,146,5.07,21,0
6,upper,3,3.09,89,3.09,0,15
7,lower,3,3.09,79,2.74,23,0
8,lower,4,4.01,62,2.15,21,0
9,upper,4,4.01,51,1.77,0,11


## Tuning the entry band

Lower vwap_entry_band to fire on closer bands and get more (but noisier) trades. \
Band 0 = 1.28σ (tightest), band 4 = 4.01σ (furthest, default). vwap_band_devs and \
vwap_session are also part of StrategyConfig if you want to retune the stdev \
ladder or use a non-daily anchor (e.g. `"h"` for hourly resets).

In [11]:
from dataclasses import replace

# Example: fire on the 2.51σ band (index 2) instead of the furthest one
tight_config = replace(STRATEGY_CONFIG, vwap_entry_band=2)
tight_strategy = VWAPBandsStrategy(tight_config)
tight_result = Backtester(tight_strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)
print(tight_result.summary())

════════════════════════════════════════════════════════════
  Backtest Summary: vwap_bands
  BTCUSDT | 15 | 2881 bars
════════════════════════════════════════════════════════════
  Total trades      : 51
  Win / Loss         : 23 / 28
  Win rate           : 45.1%
  Total P&L (bps)    : -1273.5
  Avg P&L (bps)      : -25.0
  Max win (bps)      : +163.7
  Max loss (bps)     : -385.9
  Profit factor      : 0.41
  Max drawdown (bps) : 1305.2
  Sharpe (approx)    : -0.23
  ────────────────────────────────────────
  Exits by reason:
    take_profit      : 51
════════════════════════════════════════════════════════════


## Entry-band sweep

Re-run the strategy with vwap_entry_band set to each of the five band levels in turn and collect per-run profit metrics, sorted by total P&L (bps) descending. final_balance and return_pct compound a $100 starting balance through that run's trades.

In [13]:
from dataclasses import replace
import pandas as pd

INITIAL_BALANCE = 100  # USD

sweep_rows = []
for k, mult in enumerate(STRATEGY_CONFIG.vwap_band_devs):
    cfg_k = replace(STRATEGY_CONFIG, vwap_entry_band=k)
    strat_k = VWAPBandsStrategy(cfg_k)
    r = Backtester(strat_k, symbol=SYMBOL, trading_config=TRADING_CONFIG).run(df, interval=INTERVAL)

    balance, peak, max_dd = INITIAL_BALANCE, INITIAL_BALANCE, 0.0
    for t in r.trades:
        balance *= (1 + t.pnl_bps / 10_000)
        peak = max(peak, balance)
        max_dd = max(max_dd, (peak - balance) / peak)

    sweep_rows.append({
        "entry_band": k,
        "stdev": mult,
        "trades": r.total_trades,
        "win_rate": r.win_rate,
        "total_pnl_bps": r.total_pnl_bps,
        "avg_pnl_bps": r.avg_pnl_bps,
        "profit_factor": r.profit_factor,
        "max_dd_bps": r.max_drawdown_bps,
        "final_balance": balance,
        "return_pct": (balance / INITIAL_BALANCE - 1) * 100,
    })

sweep = (
    pd.DataFrame(sweep_rows)
      .sort_values("total_pnl_bps", ascending=False)
      .reset_index(drop=True)
)
sweep["win_rate"] = (sweep["win_rate"] * 100).round(1)
sweep["total_pnl_bps"] = sweep["total_pnl_bps"].round(1)
sweep["avg_pnl_bps"] = sweep["avg_pnl_bps"].round(1)
sweep["profit_factor"] = sweep["profit_factor"].round(2)
sweep["max_dd_bps"] = sweep["max_dd_bps"].round(1)
sweep["final_balance"] = sweep["final_balance"].round(2)
sweep["return_pct"] = sweep["return_pct"].round(2)
sweep

,entry_band,stdev,trades,win_rate,total_pnl_bps,avg_pnl_bps,profit_factor,max_dd_bps,final_balance,return_pct
0,1,2.01,95,65.3,154.4,1.6,1.06,548.8,101.13,1.13
1,4,4.01,32,34.4,23.8,0.7,1.07,199.2,100.21,0.21
2,3,3.09,38,36.8,-532.0,-14.0,0.49,638.0,94.67,-5.33
3,2,2.51,51,45.1,-1273.5,-25.0,0.41,1305.2,87.76,-12.24
4,0,1.28,125,61.6,-1423.5,-11.4,0.62,1607.3,86.25,-13.75


## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart under data/live/ each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy vwap_bands \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import StrategyConfig
from engine.strategies import VWAPBandsStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = StrategyConfig()
strategy = VWAPBandsStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{strategy.name}.db"),
)

engine.run()  # blocks until Ctrl+C or kernel interrupt